# Example 7.6.1 / 1: Order Items as a Permutation

Companion notebook for **7.6.1 alldiff and CP-Style Models** in the AMPL lab Documentation, section "Order Items as a Permutation". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Order Items as a Permutation

If `card(I) = n`, each integer variable lies in `1..n`, and all values are different, then the
variables form a permutation of `1..n`. Here four items get distinct positions, and heavier items
should come first:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param n integer > 0;                  # number of items and positions
set I := 1..n;                        # items to place
param weight {I} > 0;                 # cost per position of each item
var Position {I} integer >= 1, <= n;  # position of each item

minimize WeightedPosition: sum {i in I} weight[i] * Position[i];
# all positions differ, so they form a permutation of 1..n
subject to OnePositionEach:
    alldiff {i in I} Position[i];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["n"] = 4
ampl.param["weight"] = {1: 2, 2: 7, 3: 1, 4: 5}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.var["Position"].to_dict())

HiGHS 1.14.0: optimal solution; objective 27
20 simplex iterations
1 branching nodes
{1: 3, 2: 1, 3: 4, 4: 2}


### Pure AMPL: data.dat and model.run

Item 2, the heaviest, takes position 1, then items 4, 1 and 3. Without `alldiff` every item would
take position 1.

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param n := 4;
param weight := 1 2  2 7  3 1  4 5;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Position;

HiGHS 1.14.0: optimal solution; objective 27
20 simplex iterations
1 branching nodes
Position [*] :=
1  3
2  1
3  4
4  2
;

